# 🧠 ML Workflow Cheatsheet
**Machine Learning Bible — Copy-Paste Reference Notebook**

> Replace `df` with your actual DataFrame name.  
> Replace `'target_column'` with your actual target column name.  
> Every cell is self-contained and copy-paste ready.

---
| Section | What it covers |
|---------|----------------|
| 0 | Imports |
| 1 | Load Data |
| 2 | Exploratory Data Analysis (EDA) |
| 3 | Visualize Relationships |
| 4 | Preprocessing |
| 5 | Feature / Target Split |
| 6 | Train / Test Split |
| 7 | Train Model |
| 8 | Evaluate Model |
| 9 | Improve Model |
| 10 | Feature Engineering & Importance |
| 11 | Save & Load Model |
| 12 | Key Takeaways |


---
## 0 — Imports

In [ ]:
# ─── Core ────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ─── Visualisation ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ─── Preprocessing ───────────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split

# ─── Regression Models ───────────────────────────────────────────────────────
from sklearn.linear_model import LinearRegression, Ridge, Lasso

# ─── Classification Models ───────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

# ─── Evaluation — Regression ─────────────────────────────────────────────────
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ─── Evaluation — Classification ─────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# ─── Model Persistence ───────────────────────────────────────────────────────
import pickle

# ─── Display Settings ────────────────────────────────────────────────────────
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

print('✅ All imports OK')

---
## 1 — Load Data

In [ ]:
# ─── Load from CSV ────────────────────────────────────────────────────────────
df = pd.read_csv('your_file.csv')          # ← replace with your path

# ─── Quick sanity check ──────────────────────────────────────────────────────
print('Shape:', df.shape)                   # (rows, columns)
print('\nColumn names:')
print(df.columns.tolist())
df.head()

---
## 2 — Exploratory Data Analysis (EDA)

**Goal:** Understand the data before touching it.  
Ask yourself:
- How many rows / columns?
- What are the data types?
- Are there missing values?
- What do the numbers look like (mean, std, min/max)?
- Are there obvious outliers?

In [ ]:
# ─── Shape ───────────────────────────────────────────────────────────────────
print(f'Rows: {df.shape[0]} | Columns: {df.shape[1]}')

In [ ]:
# ─── Data Types ──────────────────────────────────────────────────────────────
# object  = string/categorical → needs encoding before ML
# int64   = integer numbers
# float64 = decimal numbers
# bool    = True/False → needs to be cast to int for most models
df.dtypes

In [ ]:
# ─── Missing Values ───────────────────────────────────────────────────────────
# NaN values break most ML models — always check!
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'count': missing, 'percent': missing_pct})
missing_df[missing_df['count'] > 0].sort_values('percent', ascending=False)

In [ ]:
# ─── Descriptive Statistics ───────────────────────────────────────────────────
# Only works for numeric columns. Booleans are excluded unless cast to int.
# Key things to look for:
#   - mean vs median far apart → likely skewed distribution
#   - min/max far from mean    → likely outliers
#   - std very high            → high variance, may need scaling
df.describe()

In [ ]:
# ─── Value Counts for Categorical Columns ─────────────────────────────────────
# Check class balance for your target — imbalanced → accuracy score can be misleading!
cat_cols = df.select_dtypes(include='object').columns

for col in cat_cols:
    print(f'\n── {col} ──')
    print(df[col].value_counts())

In [ ]:
# ─── Unique Values per Column ─────────────────────────────────────────────────
# High cardinality (many unique values) in categorical columns → careful with get_dummies!
df.nunique().sort_values(ascending=False)

---
## 3 — Visualize Relationships

**Reading distributions:**

| Shape | What it means | Common in |
|-------|--------------|----------|
| 🔔 Normal (bell curve) | Values centered around mean, symmetric | Heights, IQ scores |
| 📐 Right-skewed | Long tail on right, most values are low | Income, insurance charges |
| 📐 Left-skewed | Long tail on left, most values are high | Exam scores near max |
| 📦 Bimodal | Two peaks → probably two subgroups in your data | Age + condition |

**Reading correlation heatmap:**

| Value | Meaning |
|-------|---------|
| +1.0 | Perfect positive correlation |
| +0.7 to +1.0 | Strong positive |
| +0.4 to +0.7 | Moderate positive |
| -0.4 to +0.4 | Weak / no linear relationship |
| -0.7 to -0.4 | Moderate negative |
| -1.0 | Perfect negative correlation |

In [ ]:
# ─── Histogram of All Numeric Columns ─────────────────────────────────────────
# Each histogram = distribution of one feature
# Right-skewed → consider log transformation
# Normal      → usually fine for linear models
num_cols = df.select_dtypes(include='number').columns

fig, axes = plt.subplots(len(num_cols), 1, figsize=(10, 4 * len(num_cols)))

for ax, col in zip(axes, num_cols):
    ax.hist(df[col].dropna(), bins=30, color='steelblue', edgecolor='white', alpha=0.85)
    ax.set_title(f'Distribution: {col}', fontsize=13, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# ─── Pairplot (Numeric Columns Only) ──────────────────────────────────────────
# Diagonal = distribution of each feature
# Off-diagonal = scatter plot of two features → look for linear patterns
# hue = color by target (shows separation between classes)

# ── Regression (no hue) ──
sns.pairplot(df.select_dtypes(include='number'), diag_kind='hist')

# ── Classification (color by target) — uncomment if applicable ──
# sns.pairplot(df, hue='target_column', diag_kind='hist')

plt.suptitle('Pairplot', y=1.02, fontsize=14)
plt.show()

In [ ]:
# ─── Correlation Heatmap ──────────────────────────────────────────────────────
# Only works on numeric columns!
# High correlation with target → good predictor
# High correlation between features → multicollinearity (may hurt linear models)

plt.figure(figsize=(12, 8))

corr = df.select_dtypes(include='number').corr()

mask = np.triu(np.ones_like(corr, dtype=bool))   # hide upper triangle (it's a mirror)

sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5
)

plt.title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── Boxplots — Outlier Detection ─────────────────────────────────────────────
# Box = IQR (middle 50% of data)
# Whiskers = 1.5 × IQR from box edges
# Dots beyond whiskers = potential outliers

num_cols = df.select_dtypes(include='number').columns
n = len(num_cols)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))

if n == 1:
    axes = [axes]

for ax, col in zip(axes, num_cols):
    ax.boxplot(df[col].dropna())
    ax.set_title(col, fontsize=12)

plt.suptitle('Boxplots — Outlier Detection', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── Target Distribution by Categorical Feature ───────────────────────────────
# Replace 'categorical_col' and 'target_column' as needed
# Useful for: seeing if target differs between groups

categorical_col = 'categorical_col'   # ← replace
target_col      = 'target_column'     # ← replace

plt.figure(figsize=(10, 5))

for group in df[categorical_col].unique():
    subset = df[df[categorical_col] == group][target_col]
    subset.hist(bins=30, alpha=0.6, label=str(group), edgecolor='white')

plt.title(f'{target_col} distribution by {categorical_col}', fontsize=13, fontweight='bold')
plt.xlabel(target_col)
plt.ylabel('Count')
plt.legend(title=categorical_col)
plt.tight_layout()
plt.show()

---
## 4 — Preprocessing

**Order matters:** Always preprocess → split → scale (never scale before splitting!)

| Step | Why |
|------|-----|
| Handle missing values | NaN crashes most models |
| Encode categoricals | ML models need numbers, not strings |
| Scale features | KNN, LogReg, SVM are distance-based → scale! Decision Trees do NOT need scaling |

### When to use what for missing values?

| Strategy | When to use |
|----------|--------------|
| `fillna(median)` | Numeric, outliers present |
| `fillna(mean)` | Numeric, normally distributed |
| `fillna(mode)` | Categorical |
| `fillna('Unknown')` | Categorical, unknown is meaningful |
| `dropna()` | Very few rows affected (< 2%) |

### When to use what for encoding?

| Method | When to use |
|--------|-------------|
| `pd.get_dummies()` | Quick, in a notebook, few categories |
| `OneHotEncoder` | Production pipeline, many categories, cross-validation |
| `LabelEncoder` | Ordinal categories (small, medium, large) |
| `OrdinalEncoder` | Multiple ordinal columns at once |

In [ ]:
# ─── Handle Missing Values ────────────────────────────────────────────────────

# Option A: Fill numeric with median (robust to outliers)
df['numeric_col'] = df['numeric_col'].fillna(df['numeric_col'].median())

# Option B: Fill numeric with mean
df['numeric_col'] = df['numeric_col'].fillna(df['numeric_col'].mean())

# Option C: Fill categorical with mode (most frequent value)
df['cat_col'] = df['cat_col'].fillna(df['cat_col'].mode()[0])

# Option D: Fill categorical with a fixed string
df['cat_col'] = df['cat_col'].fillna('Unknown')

# Option E: Drop rows with NaN
df = df.dropna()

# Verify: no more missing values
print('Missing values left:', df.isnull().sum().sum())

In [ ]:
# ─── Encode Categorical Columns ───────────────────────────────────────────────
# pd.get_dummies creates one binary column per category
# drop_first=True removes one column per group to avoid multicollinearity
# dtype=int ensures 0/1 instead of True/False (which breaks df.describe())

cat_cols_to_encode = ['cat_col_1', 'cat_col_2']   # ← replace

df = pd.get_dummies(df, columns=cat_cols_to_encode, drop_first=False, dtype=int)

print('Columns after encoding:')
print(df.columns.tolist())

In [ ]:
# ─── Remove Duplicates ────────────────────────────────────────────────────────
print('Duplicates before:', df.duplicated().sum())
df = df.drop_duplicates()
print('Duplicates after: ', df.duplicated().sum())

---
## 5 — Feature / Target Split

> **X** = features (everything the model learns from)  
> **y** = target (what we want to predict)

In [ ]:
# ─── Split X and y ────────────────────────────────────────────────────────────
target_column = 'target_column'   # ← replace with your target

X = df.drop(columns=[target_column])
y = df[target_column]

print('X shape:', X.shape)
print('y shape:', y.shape)
print('\nFeatures:')
print(X.columns.tolist())

---
## 6 — Train / Test Split

> The model must **never** see the test data during training.  
> `random_state=42` makes the split reproducible.  
> Standard split: **80% train, 20% test** (`test_size=0.2`)

In [ ]:
# ─── Train / Test Split ───────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% test, 80% train
    random_state=42       # reproducibility
)

print(f'Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')

In [ ]:
# ─── Feature Scaling ──────────────────────────────────────────────────────────
# ⚠️  ALWAYS fit scaler on X_train ONLY, then transform both train and test
#     Fitting on test data = data leakage = artificially good results
#
# Use StandardScaler when:
#   - Algorithm is distance-based: KNN, SVM, Logistic Regression, Linear Regression
#   - Data is roughly normally distributed
#
# Use MinMaxScaler when:
#   - You need values in a fixed range [0, 1]
#   - Neural networks, image pixel values
#
# Skip scaling for:
#   - Decision Trees, Random Forests (tree-based models don't care about scale)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)   # fit + transform on train
X_test_scaled  = scaler.transform(X_test)         # transform only on test

print('Scaling done. X_train_scaled shape:', X_train_scaled.shape)

---
## 7 — Train Model

### Which algorithm to choose?

| Problem | Algorithm | Notes |
|---------|-----------|-------|
| Regression | `LinearRegression` | Baseline, fast, interpretable |
| Regression | `Ridge / Lasso` | LinearRegression + regularisation (prevents overfitting) |
| Classification | `LogisticRegression` | Binary/multi, output = probability |
| Classification | `KNeighborsClassifier` | Simple, no assumptions, needs scaling |
| Both | `DecisionTreeClassifier / Regressor` | Interpretable, no scaling needed, prone to overfit |
| Both | `SVC / SVR` | Good for small-medium datasets, needs scaling |

> **Golden Rule:** Start simple. LinearRegression or LogisticRegression first — then improve.

In [ ]:
# ─── Regression ───────────────────────────────────────────────────────────────
model = LinearRegression()
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

print('Model trained ✅')
print(f'Intercept:    {model.intercept_:.4f}')
print(f'Coefficients: {model.coef_}')

In [ ]:
# ─── Classification — Logistic Regression ────────────────────────────────────
model_lr = LogisticRegression(max_iter=1000, random_state=42)
model_lr.fit(X_train_scaled, y_train)

y_pred_lr = model_lr.predict(X_test_scaled)
print('Logistic Regression trained ✅')

In [ ]:
# ─── Classification — KNN ─────────────────────────────────────────────────────
# Rule of thumb: k ≈ sqrt(n_samples), always odd for binary classification
k = 5
model_knn = KNeighborsClassifier(n_neighbors=k)
model_knn.fit(X_train_scaled, y_train)

y_pred_knn = model_knn.predict(X_test_scaled)
print(f'KNN (k={k}) trained ✅')

In [ ]:
# ─── Classification — Decision Tree ──────────────────────────────────────────
# max_depth limits tree size → prevents overfitting
# No scaling needed for Decision Trees!
model_dt = DecisionTreeClassifier(max_depth=5, random_state=42)
model_dt.fit(X_train, y_train)            # use X_train (unscaled)

y_pred_dt = model_dt.predict(X_test)      # use X_test (unscaled)
print('Decision Tree trained ✅')

---
## 8 — Evaluate Model

### Regression Metrics

| Metric | Formula | Interpretation | Good when |
|--------|---------|---------------|----------|
| **R²** | 1 - SS_res/SS_tot | % of variance explained. 1.0 = perfect, 0 = useless | Always check first |
| **MAE** | mean(|y - ŷ|) | Average error in same units as target | Outliers present |
| **RMSE** | √mean((y - ŷ)²) | Like MAE but penalises large errors more | Outliers matter a lot |

### Classification Metrics

| Metric | Interpretation | Good when |
|--------|---------------|----------|
| **Accuracy** | % correct predictions overall | Classes are balanced |
| **Precision** | Of all predicted positive — how many were truly positive? | False positives are costly (e.g. spam filter) |
| **Recall** | Of all actual positive — how many did we catch? | False negatives are costly (e.g. cancer detection) |
| **F1-Score** | Harmonic mean of Precision & Recall | Imbalanced classes |
| **Confusion Matrix** | Visual breakdown of TP/FP/TN/FN | Always look at this |

In [ ]:
# ─── Regression Evaluation ────────────────────────────────────────────────────
r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print('── Regression Metrics ──────────────────')
print(f'R²   (higher is better): {r2:.4f}')
print(f'MAE  (lower is better):  {mae:.2f}')
print(f'RMSE (lower is better):  {rmse:.2f}')

# ── Train vs Test R² — check for overfitting ──
r2_train = r2_score(y_train, model.predict(X_train_scaled))
r2_test  = r2_score(y_test,  model.predict(X_test_scaled))
print(f'\nR² Train: {r2_train:.4f}  |  R² Test: {r2_test:.4f}')
print('→ If train >> test: overfitting. If both low: underfitting.')

In [ ]:
# ─── Regression: Predicted vs Actual Plot ─────────────────────────────────────
# Points close to the diagonal = good predictions
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred, alpha=0.5, color='steelblue', edgecolors='white')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Predicted vs Actual', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── Classification Evaluation ────────────────────────────────────────────────
# Use y_pred from whichever model you want to evaluate
y_pred_eval = y_pred_lr   # ← swap to y_pred_knn or y_pred_dt as needed

print('── Classification Metrics ──────────────────')
print(f'Accuracy: {accuracy_score(y_test, y_pred_eval):.4f}')
print()
print(classification_report(y_test, y_pred_eval))

In [ ]:
# ─── Confusion Matrix ─────────────────────────────────────────────────────────
#
#          Predicted
#          Neg   Pos
# Actual  [ TN  | FP ]
#         [ FN  | TP ]
#
# TP = True Positive  → predicted positive, was positive
# TN = True Negative  → predicted negative, was negative
# FP = False Positive → predicted positive, was negative (Type I error)
# FN = False Negative → predicted negative, was positive (Type II error)

cm = confusion_matrix(y_test, y_pred_eval)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── Model Comparison Table ───────────────────────────────────────────────────
# Use this to compare multiple classification models side by side

results = {
    'Logistic Regression': accuracy_score(y_test, y_pred_lr),
    'KNN (k=5)':           accuracy_score(y_test, y_pred_knn),
    'Decision Tree':       accuracy_score(y_test, y_pred_dt),
}

results_df = pd.DataFrame.from_dict(
    results, orient='index', columns=['Accuracy']
).sort_values('Accuracy', ascending=False)

results_df['Accuracy'] = results_df['Accuracy'].map('{:.4f}'.format)
print(results_df)

---
## 9 — Improve Model

### Diagnosing problems

| Symptom | Problem | Fix |
|---------|---------|-----|
| Train acc high, test acc low | **Overfitting** | Reduce complexity (less depth, higher k, regularisation) |
| Both train and test acc low | **Underfitting** | Add features, reduce regularisation, try more complex model |
| Accuracy high but F1 low | **Class imbalance** | Check confusion matrix, use F1 instead of accuracy |
| R² low | **Bad features or wrong model** | Feature engineering, try different algorithm |

In [ ]:
# ─── KNN: Find Optimal K ──────────────────────────────────────────────────────
# Train with k = 1 to 20, plot accuracy → pick the k at the elbow

k_range  = range(1, 21)
k_scores = []

for k in k_range:
    knn_tmp = KNeighborsClassifier(n_neighbors=k)
    knn_tmp.fit(X_train_scaled, y_train)
    k_scores.append(accuracy_score(y_test, knn_tmp.predict(X_test_scaled)))

best_k = k_range[k_scores.index(max(k_scores))]

plt.figure(figsize=(9, 4))
plt.plot(k_range, k_scores, marker='o', color='steelblue')
plt.axvline(best_k, color='red', linestyle='--', label=f'Best k = {best_k}')
plt.xlabel('k')
plt.ylabel('Accuracy')
plt.title('KNN: Accuracy vs k', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Best k: {best_k} → Accuracy: {max(k_scores):.4f}')

In [ ]:
# ─── Decision Tree: Find Optimal max_depth ────────────────────────────────────

depth_range  = range(1, 21)
train_scores = []
test_scores  = []

for d in depth_range:
    dt_tmp = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt_tmp.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, dt_tmp.predict(X_train)))
    test_scores.append(accuracy_score(y_test,  dt_tmp.predict(X_test)))

plt.figure(figsize=(9, 4))
plt.plot(depth_range, train_scores, label='Train',  marker='o', color='steelblue')
plt.plot(depth_range, test_scores,  label='Test',   marker='s', color='coral')
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('Decision Tree: Train vs Test Accuracy', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

best_depth = depth_range[test_scores.index(max(test_scores))]
print(f'Best depth: {best_depth} → Test Accuracy: {max(test_scores):.4f}')

In [ ]:
# ─── Ridge / Lasso Regression (regularised linear regression) ─────────────────
# Ridge (L2): shrinks all coefficients, keeps all features
# Lasso (L1): can zero out coefficients → built-in feature selection
# alpha: regularisation strength — higher = more regularisation

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)
r2_ridge = r2_score(y_test, ridge.predict(X_test_scaled))
print(f'Ridge R²: {r2_ridge:.4f}')

lasso = Lasso(alpha=0.1)
lasso.fit(X_train_scaled, y_train)
r2_lasso = r2_score(y_test, lasso.predict(X_test_scaled))
print(f'Lasso R²: {r2_lasso:.4f}')

---
## 10 — Feature Engineering & Importance

> Feature Engineering = creating **new features** from existing ones.  
> Goal: give the model more signal to learn from.

### Common techniques

| Technique | Example | Why |
|-----------|---------|-----|
| Interaction feature | `bmi × smoker` | Captures combined effect |
| Ratio | `charges / age` | Normalises by another feature |
| Bin / Bucket | `age_group = pd.cut(age, bins=[...])` | Non-linear patterns |
| Log transform | `np.log1p(charges)` | Fix right-skewed distribution |
| Binary flag | `is_obese = (bmi >= 30).astype(int)` | Threshold-based signal |

In [ ]:
# ─── Feature Engineering Examples ────────────────────────────────────────────

# Interaction feature
df['feature_interaction'] = df['col_a'] * df['col_b']

# Binary flag
df['is_high_value'] = (df['numeric_col'] > df['numeric_col'].median()).astype(int)

# Binning / age groups
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 25, 40, 60, 100],
    labels=['young', 'adult', 'middle_aged', 'senior']
)

# Log transform (use on right-skewed target / features)
# np.log1p handles 0 values safely (log(1 + x))
df['log_target'] = np.log1p(df['target_column'])

print('Feature engineering done.')
df.head()

In [ ]:
# ─── Feature Importance (Decision Tree / Tree-based models) ───────────────────
# Higher value = more important feature for the model's decisions

feat_imp = pd.Series(model_dt.feature_importances_, index=X.columns)
feat_imp = feat_imp.sort_values(ascending=False)

plt.figure(figsize=(10, 5))
feat_imp.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Feature Importance (Decision Tree)', fontsize=13, fontweight='bold')
plt.xlabel('Feature')
plt.ylabel('Importance')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print('\nTop 5 features:')
print(feat_imp.head())

In [ ]:
# ─── Feature Importance (Linear / Logistic Regression) ───────────────────────
# Coefficients = how much each feature influences the prediction
# Works best AFTER scaling (so units are comparable)

coef_series = pd.Series(
    np.abs(model_lr.coef_[0]),    # absolute value for importance
    index=X.columns
).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
coef_series.plot(kind='bar', color='coral', edgecolor='white')
plt.title('Feature Importance (Logistic Regression coefficients)', fontsize=13, fontweight='bold')
plt.xlabel('Feature')
plt.ylabel('|Coefficient|')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## 11 — Save & Load Model

In [ ]:
# ─── Save Model to Disk ───────────────────────────────────────────────────────
with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)

print('Model saved as model.pkl ✅')

In [ ]:
# ─── Load Model from Disk ─────────────────────────────────────────────────────
with open('model.pkl', 'rb') as f:
    model_loaded = pickle.load(f)

# Test the loaded model
print('Model loaded ✅')
print('Score on test set:', model_loaded.score(X_test_scaled, y_test))

---
## 12 — Key Takeaways

```
╔══════════════════════════════════════════════════════════════════╗
║               ML WORKFLOW — DECISION GUIDE                      ║
╠══════════════════════════════════════════════════════════════════╣
║  Predicting a number?        → Regression                       ║
║  Predicting a category?      → Classification                   ║
╠══════════════════════════════════════════════════════════════════╣
║  Need scaling?                                                   ║
║    KNN, SVM, LogReg, LinReg  → YES (StandardScaler)             ║
║    Decision Trees, RF        → NO                               ║
╠══════════════════════════════════════════════════════════════════╣
║  Which metric?                                                   ║
║    Regression                → R², MAE, RMSE                    ║
║    Balanced classes          → Accuracy                         ║
║    Imbalanced classes        → F1-Score, Precision, Recall      ║
║    Always                    → Confusion Matrix                  ║
╠══════════════════════════════════════════════════════════════════╣
║  Overfitting?  Train >> Test → Reduce complexity                ║
║  Underfitting? Both low      → Add features / more complex model║
╠══════════════════════════════════════════════════════════════════╣
║  NEVER fit scaler on test data                (data leakage!)   ║
║  NEVER evaluate on training data              (overfitting lie!)║
║  ALWAYS start with the simplest model first   (KISS principle)  ║
╚══════════════════════════════════════════════════════════════════╝
```

### Scikit-Learn Universal API

```python
model.fit(X_train, y_train)       # train
model.predict(X_test)             # predict
model.score(X_test, y_test)       # evaluate
model.transform(X)                # preprocessing
model.fit_transform(X)            # fit + transform in one step
```

> Same API for every algorithm. Swap models by changing **one line**.

---
*ML Cheatsheet — Machine Learning Bible | Dataset-agnostic reference*